In [ ]:
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Please enable GPU in Colab settings.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
from datasets import load_dataset, Dataset
import random

print("Loading English-Hindi dataset...")

raw_dataset = load_dataset(
    "cfilt/iitb-english-hindi",
    split="train",
    streaming=True
)

# Shuffle the streaming dataset
raw_dataset = raw_dataset.shuffle(
    seed=42,
    buffer_size=100000
)

# Load a manageable subset
samples = list(raw_dataset.take(55000))

print("Downloaded samples:", len(samples))

# Clean the dataset
clean_data = []

for item in samples:
    translation = item["translation"]

    english = translation["en"].strip()
    hindi = translation["hi"].strip()

    if (
        len(english) > 2
        and len(hindi) > 2
        and len(english) <= 300
        and len(hindi) <= 300
    ):
        clean_data.append({
            "en": english,
            "hi": hindi
        })

dataset = Dataset.from_list(clean_data)

print("Clean dataset size:", len(dataset))
print("\nExample:")
print("English:", dataset[0]["en"])
print("Hindi:", dataset[0]["hi"])

# Split into training and temporary sets
split_1 = dataset.train_test_split(
    test_size=0.10,
    seed=42
)

# Split temporary set into validation and test
split_2 = split_1["test"].train_test_split(
    test_size=0.50,
    seed=42
)

train_dataset = split_1["train"]
validation_dataset = split_2["train"]
test_dataset = split_2["test"]

print("\nDataset split:")
print("Training:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Testing:", len(test_dataset))

Loading English-Hindi dataset...


README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

Downloaded samples: 55000
Clean dataset size: 54583

Example:
English: Import from version control system
Hindi: संस्करण नियंत्रण सिस्टम प्लगिन

Dataset split:
Training: 49124
Validation: 2729
Testing: 2730


In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"

print("Loading tokenizer...")

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)

print("Loading Transformer model...")

model = MarianMTModel.from_pretrained(MODEL_NAME)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("\nModel loaded successfully!")
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


... (5 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  306MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


Model loaded successfully!
Device: cuda
Parameters: 76381184


In [ ]:
MAX_LENGTH = 96

def preprocess_function(batch):

    inputs = batch["en"]
    targets = batch["hi"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


print("Tokenizing training data...")

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

print("Tokenizing validation data...")

tokenized_validation = validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=validation_dataset.column_names
)

print("Tokenizing test data...")

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

print("\nTokenization completed!")

print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_validation))
print("Testing examples:", len(tokenized_test))

Tokenizing training data...


Map:   0%|          | 0/49124 [00:00<?, ? examples/s]

Tokenizing validation data...


Map:   0%|          | 0/2729 [00:00<?, ? examples/s]

Tokenizing test data...


Map:   0%|          | 0/2730 [00:00<?, ? examples/s]


Tokenization completed!
Training examples: 49124
Validation examples: 2729
Testing examples: 2730


In [ ]:
import numpy as np
import sacrebleu

from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

# BLEU evaluation
def compute_metrics(eval_preds):

    predictions, labels = eval_preds

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.where(
        predictions != -100,
        predictions,
        tokenizer.pad_token_id
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_predictions = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    decoded_predictions = [
        prediction.strip()
        for prediction in decoded_predictions
    ]

    decoded_labels = [
        label.strip()
        for label in decoded_labels
    ]

    bleu = sacrebleu.corpus_bleu(
        decoded_predictions,
        [decoded_labels]
    )

    return {
        "bleu": bleu.score
    }


training_args = Seq2SeqTrainingArguments(

    output_dir="./english_hindi_transformer",

    # Training configuration
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,

    max_steps=1200,

    # Evaluation
    eval_strategy="steps",
    eval_steps=300,

    # Checkpoints
    save_strategy="steps",
    save_steps=300,
    save_total_limit=2,

    # Performance
    fp16=torch.cuda.is_available(),
    predict_with_generate=True,
    generation_max_length=96,
    generation_num_beams=2,

    # Logging
    logging_steps=50,
    report_to="none",

    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_bleu",
    greater_is_better=True,

    # Avoid unnecessary reporting
    disable_tqdm=False
)


trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_validation,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


print("Starting Transformer fine-tuning...")

train_result = trainer.train()

print("\nTraining completed!")

print("Training results:")
print(train_result.metrics)

Starting Transformer fine-tuning...


Step,Training Loss,Validation Loss,Bleu
300,1.300140,0.575253,82.943204
600,1.323999,0.506960,83.855848
900,1.185314,0.473057,84.659148
1200,1.119394,0.462102,84.924686


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].



Training completed!
Training results:
{'train_runtime': 575.9655, 'train_samples_per_second': 33.335, 'train_steps_per_second': 2.083, 'total_flos': 84017097474048.0, 'train_loss': 1.2598649501800536, 'epoch': 0.39081582804103565}


In [ ]:
print("Evaluating model on test dataset...")

test_results = trainer.evaluate(
    eval_dataset=tokenized_test,
    metric_key_prefix="test"
)

print("\nFinal Test Results:")

for key, value in test_results.items():
    print(f"{key}: {value}")

Evaluating model on test dataset...


Training Loss,Validation Loss,Step,Bleu
1.119394,0.428088,1200,85.344162



Final Test Results:
test_loss: 0.42808765172958374
test_bleu: 85.34416179720527


In [ ]:
def translate_to_hindi(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=96,
        truncation=True
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_length=96,
            num_beams=4,
            early_stopping=True
        )

    translated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return translated_text


test_sentences = [
    "Hello, how are you?",
    "Artificial intelligence is changing the world.",
    "India is a beautiful country.",
    "Machine learning is an important part of artificial intelligence.",
    "Education is the key to success."
]

print("ENGLISH TO HINDI TRANSLATION RESULTS\n")

for sentence in test_sentences:

    hindi_translation = translate_to_hindi(sentence)

    print("English:", sentence)
    print("Hindi:", hindi_translation)
    print("-" * 60)

ENGLISH TO HINDI TRANSLATION RESULTS

English: Hello, how are you?
Hindi: हैलो, तुम कैसे हो?
------------------------------------------------------------
English: Artificial intelligence is changing the world.
Hindi: तकनीकी बुद्धि दुनिया बदल रहा है।
------------------------------------------------------------
English: India is a beautiful country.
Hindi: भारत एक सुंदर देश है.
------------------------------------------------------------
English: Machine learning is an important part of artificial intelligence.
Hindi: मशीन सीखने के लिए कृत्रिम बुद्धि का एक महत्वपूर्ण हिस्सा है।
------------------------------------------------------------
English: Education is the key to success.
Hindi: शिक्षा सफलता की कुंजी है ।
------------------------------------------------------------


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive")

SAVE_PATH = "/content/drive/MyDrive/English_Hindi_Transformer"

os.makedirs(SAVE_PATH, exist_ok=True)

# Save model
trainer.save_model(SAVE_PATH)

# Save tokenizer
tokenizer.save_pretrained(SAVE_PATH)

# Save training results
with open(
    os.path.join(SAVE_PATH, "training_results.txt"),
    "w",
    encoding="utf-8"
) as file:

    file.write("English to Hindi Transformer\n")
    file.write("============================\n\n")

    for key, value in test_results.items():
        file.write(f"{key}: {value}\n")

print("\nModel saved successfully!")
print("Location:", SAVE_PATH)

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully!
Location: /content/drive/MyDrive/English_Hindi_Transformer


In [10]:
import os

SAVE_PATH = "/content/drive/MyDrive/English_Hindi_Transformer"

print("Files saved in Google Drive:\n")

for file in os.listdir(SAVE_PATH):
    file_path = os.path.join(SAVE_PATH, file)

    if os.path.isfile(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"{file} — {size_mb:.2f} MB")

Files saved in Google Drive:

config.json — 0.00 MB
generation_config.json — 0.00 MB
model.safetensors — 289.64 MB
tokenizer_config.json — 0.00 MB
vocab.json — 2.12 MB
source.spm — 0.77 MB
target.spm — 1.02 MB
training_args.bin — 0.01 MB
training_results.txt — 0.00 MB


In [11]:
from datasets import DatasetDict
import os

DATA_PATH = "/content/drive/MyDrive/English_Hindi_Dataset"

os.makedirs(DATA_PATH, exist_ok=True)

dataset_splits = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset
})

dataset_splits.save_to_disk(DATA_PATH)

print("Dataset splits saved successfully!")
print("Location:", DATA_PATH)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

Saving the dataset (0/1 shards):   0%|          | 0/49124 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2729 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2730 [00:00<?, ? examples/s]

Dataset splits saved successfully!
Location: /content/drive/MyDrive/English_Hindi_Dataset

Dataset sizes:
Train: 49124
Validation: 2729
Test: 2730


In [12]:
!pip install -q huggingface_hub bitsandbytes accelerate

In [13]:
from huggingface_hub import login

login()

In [14]:
from huggingface_hub import model_info

models = [
    "google/mt5-small",
    "google/gemma-2-2b-it",
    "meta-llama/Llama-3.2-1B-Instruct"
]

for name in models:
    try:
        info = model_info(name)
        print("ACCESS OK:", name)
    except Exception as e:
        print("ACCESS FAILED:", name)
        print(str(e)[:250])

ACCESS OK: google/mt5-small
ACCESS OK: google/gemma-2-2b-it
ACCESS OK: meta-llama/Llama-3.2-1B-Instruct


In [15]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

GEMMA_MODEL = "google/gemma-2-2b-it"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Gemma tokenizer...")

gemma_tokenizer = AutoTokenizer.from_pretrained(
    GEMMA_MODEL
)

print("Loading Gemma model...")

gemma_model = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

print("Gemma loaded successfully!")

Loading Gemma tokenizer...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Gemma loaded successfully!


In [16]:
def gemma_translate(text):

    messages = [
        {
            "role": "user",
            "content": (
                "Translate the following English sentence into Hindi. "
                "Return only the Hindi translation.\n\n"
                + text
            )
        }
    ]

    prompt = gemma_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = gemma_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(gemma_model.device)

    with torch.no_grad():
        outputs = gemma_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return gemma_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )


print(gemma_translate("India is a beautiful country."))

भारत एक सुन्दर देश है। 



In [17]:
import torch
import re
import time
from collections import Counter
from torch.utils.data import Dataset, DataLoader

# Reproducibility
torch.manual_seed(42)

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# Special tokens
PAD_TOKEN = "<pad>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

MAX_LENGTH = 64
MAX_VOCAB_SIZE = 16000

# Use a manageable training subset
basic_train = train_dataset.select(
    range(min(20000, len(train_dataset)))
)

basic_validation = validation_dataset
basic_test = test_dataset

print("Training examples:", len(basic_train))
print("Validation examples:", len(basic_validation))
print("Testing examples:", len(basic_test))


# Basic whitespace tokenizer
def tokenize(text):
    return text.strip().split()


# Build shared vocabulary
word_counts = Counter()

for item in basic_train:
    word_counts.update(tokenize(item["en"]))
    word_counts.update(tokenize(item["hi"]))


# Initialize vocabulary with special tokens
vocab = {
    PAD_TOKEN: PAD_IDX,
    BOS_TOKEN: BOS_IDX,
    EOS_TOKEN: EOS_IDX,
    UNK_TOKEN: UNK_IDX
}

# Add most frequent words
for word, count in word_counts.most_common(
    MAX_VOCAB_SIZE - len(vocab)
):
    if word not in vocab:
        vocab[word] = len(vocab)

# Reverse vocabulary
idx_to_word = {
    index: word
    for word, index in vocab.items()
}

VOCAB_SIZE = len(vocab)

print("\nVocabulary created!")
print("Vocabulary size:", VOCAB_SIZE)


# Convert sentence to token IDs
def encode_sentence(text):

    words = tokenize(text)

    ids = [BOS_IDX]

    for word in words:
        ids.append(vocab.get(word, UNK_IDX))

    ids.append(EOS_IDX)

    return ids[:MAX_LENGTH]


# Convert token IDs back to sentence
def decode_sentence(ids):

    words = []

    for idx in ids:

        if idx == EOS_IDX:
            break

        if idx in (PAD_IDX, BOS_IDX):
            continue

        words.append(idx_to_word.get(idx, UNK_TOKEN))

    return " ".join(words)


# Custom PyTorch Dataset
class TranslationDataset(Dataset):

    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):

        item = self.dataset[index]

        source = encode_sentence(item["en"])
        target = encode_sentence(item["hi"])

        return (
            torch.tensor(source, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )


# Prepare datasets
basic_train_encoded = TranslationDataset(basic_train)

basic_validation_encoded = TranslationDataset(
    basic_validation
)

basic_test_encoded = TranslationDataset(
    basic_test
)


print("\nDataset encoding completed!")

# Show an example
sample = basic_train[0]

print("\nOriginal English:", sample["en"])
print("Original Hindi:", sample["hi"])

print("English token IDs:", encode_sentence(sample["en"]))
print("Hindi token IDs:", encode_sentence(sample["hi"]))

Using device: cuda
Training examples: 20000
Validation examples: 2729
Testing examples: 2730

Vocabulary created!
Vocabulary size: 16000

Dataset encoding completed!

Original English: _ Program:
Original Hindi: प्रोग्रामः (_ P)
English token IDs: [1, 5, 4477, 2]
Hindi token IDs: [1, 6264, 8, 288, 2]


In [23]:
import gc
import torch

if "gemma_model" in globals():
    del gemma_model

if "gemma_tokenizer" in globals():
    del gemma_tokenizer

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleanup completed!")

GPU memory cleanup completed!


In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import os
import json
import math
import random
import numpy as np

from collections import Counter
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Configuration
MAX_LENGTH = 64
MAX_VOCAB_SIZE = 16000
BATCH_SIZE = 32

# Select manageable subsets
basic_train = train_dataset.select(
    range(min(20000, len(train_dataset)))
)

basic_validation = validation_dataset.select(
    range(min(1000, len(validation_dataset)))
)

basic_test = test_dataset.select(
    range(min(1000, len(test_dataset)))
)

print("\nDataset sizes:")
print("Training:", len(basic_train))
print("Validation:", len(basic_validation))
print("Testing:", len(basic_test))

# Special tokens
PAD_TOKEN = "<pad>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

# Whitespace tokenization
def tokenize(text):
    return text.strip().split()

# Build vocabulary using training data only
word_counts = Counter()

for item in basic_train:
    word_counts.update(tokenize(item["en"]))
    word_counts.update(tokenize(item["hi"]))

vocab = {
    PAD_TOKEN: PAD_IDX,
    BOS_TOKEN: BOS_IDX,
    EOS_TOKEN: EOS_IDX,
    UNK_TOKEN: UNK_IDX
}

for word, count in word_counts.most_common(
    MAX_VOCAB_SIZE - 4
):
    if word not in vocab:
        vocab[word] = len(vocab)

idx_to_word = {
    idx: word for word, idx in vocab.items()
}

VOCAB_SIZE = len(vocab)

print("\nVocabulary size:", VOCAB_SIZE)

# Encode sentences
def encode_sentence(text):

    words = tokenize(text)

    ids = [BOS_IDX]

    ids.extend([
        vocab.get(token, UNK_IDX)
        for token in words[:MAX_LENGTH - 2]
    ])

    ids.append(EOS_IDX)

    return ids


# Decode sentences
def decode_sentence(ids):

    words = []

    for idx in ids:

        if idx == EOS_IDX:
            break

        if idx in (PAD_IDX, BOS_IDX):
            continue

        words.append(
            idx_to_word.get(int(idx), UNK_TOKEN)
        )

    return " ".join(words)


# PyTorch dataset
class BasicTranslationDataset(Dataset):

    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):

        item = self.dataset[index]

        source = encode_sentence(item["en"])
        target = encode_sentence(item["hi"])

        return (
            torch.tensor(source, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )


train_encoded = BasicTranslationDataset(basic_train)
validation_encoded = BasicTranslationDataset(basic_validation)
test_encoded = BasicTranslationDataset(basic_test)


# Padding function
def collate_fn(batch):

    sources, targets = zip(*batch)

    sources = nn.utils.rnn.pad_sequence(
        sources,
        batch_first=True,
        padding_value=PAD_IDX
    )

    targets = nn.utils.rnn.pad_sequence(
        targets,
        batch_first=True,
        padding_value=PAD_IDX
    )

    return sources, targets


train_loader = DataLoader(
    train_encoded,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

validation_loader = DataLoader(
    validation_encoded,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_encoded,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

print("\nData preparation completed!")
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

print("\nExample:")
print("English:", basic_train[0]["en"])
print("Hindi:", basic_train[0]["hi"])
print("Encoded English:", encode_sentence(basic_train[0]["en"]))
print("Encoded Hindi:", encode_sentence(basic_train[0]["hi"]))

Device: cuda
GPU: Tesla T4

Dataset sizes:
Training: 20000
Validation: 1000
Testing: 1000

Vocabulary size: 16000

Data preparation completed!
Training batches: 625
Validation batches: 32
Test batches: 32

Example:
English: _ Program:
Hindi: प्रोग्रामः (_ P)
Encoded English: [1, 5, 4477, 2]
Encoded Hindi: [1, 6264, 8, 288, 2]


In [25]:
# Model configuration
D_MODEL = 256
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3
D_FF = 512
DROPOUT = 0.1


# Positional Embedding
class PositionalEmbedding(nn.Module):

    def __init__(self, d_model, max_length):

        super().__init__()

        self.embedding = nn.Embedding(
            max_length,
            d_model
        )

    def forward(self, x):

        batch_size, seq_length = x.shape

        positions = torch.arange(
            seq_length,
            device=x.device
        ).unsqueeze(0)

        return self.embedding(positions)


# Multi-Head Attention
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):

        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

        self.output = nn.Linear(d_model, d_model)

    def forward(
        self,
        query,
        key,
        value,
        key_padding_mask=None,
        causal_mask=None
    ):

        batch_size = query.size(0)

        q = self.query(query)
        k = self.key(key)
        v = self.value(value)

        # Split into attention heads
        q = q.view(
            batch_size, -1,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size, -1,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size, -1,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        # Scaled dot-product attention
        scores = torch.matmul(
            q, k.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        if key_padding_mask is not None:

            scores = scores.masked_fill(
                key_padding_mask[:, None, None, :],
                float("-inf")
            )

        if causal_mask is not None:

            scores = scores.masked_fill(
                causal_mask[None, None, :, :],
                float("-inf")
            )

        attention_weights = torch.softmax(
            scores, dim=-1
        )

        attention_output = torch.matmul(
            attention_weights, v
        )

        attention_output = attention_output.transpose(
            1, 2
        ).contiguous().view(
            batch_size, -1, self.d_model
        )

        return self.output(attention_output)


# Feed Forward Network
class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.network(x)


# Encoder Layer
class EncoderLayer(nn.Module):

    def __init__(self):

        super().__init__()

        self.attention = MultiHeadAttention(
            D_MODEL, NUM_HEADS
        )

        self.feed_forward = FeedForward(
            D_MODEL, D_FF, DROPOUT
        )

        self.norm1 = nn.LayerNorm(D_MODEL)
        self.norm2 = nn.LayerNorm(D_MODEL)

        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x, source_padding_mask):

        attention_output = self.attention(
            x, x, x,
            key_padding_mask=source_padding_mask
        )

        x = self.norm1(
            x + self.dropout(attention_output)
        )

        ff_output = self.feed_forward(x)

        x = self.norm2(
            x + self.dropout(ff_output)
        )

        return x


# Decoder Layer
class DecoderLayer(nn.Module):

    def __init__(self):

        super().__init__()

        self.self_attention = MultiHeadAttention(
            D_MODEL, NUM_HEADS
        )

        self.cross_attention = MultiHeadAttention(
            D_MODEL, NUM_HEADS
        )

        self.feed_forward = FeedForward(
            D_MODEL, D_FF, DROPOUT
        )

        self.norm1 = nn.LayerNorm(D_MODEL)
        self.norm2 = nn.LayerNorm(D_MODEL)
        self.norm3 = nn.LayerNorm(D_MODEL)

        self.dropout = nn.Dropout(DROPOUT)

    def forward(
        self,
        x,
        encoder_output,
        target_padding_mask,
        source_padding_mask,
        causal_mask
    ):

        # Masked self-attention
        self_attention_output = self.self_attention(
            x, x, x,
            key_padding_mask=target_padding_mask,
            causal_mask=causal_mask
        )

        x = self.norm1(
            x + self.dropout(self_attention_output)
        )

        # Encoder-decoder cross-attention
        cross_attention_output = self.cross_attention(
            x,
            encoder_output,
            encoder_output,
            key_padding_mask=source_padding_mask
        )

        x = self.norm2(
            x + self.dropout(cross_attention_output)
        )

        # Feed-forward network
        ff_output = self.feed_forward(x)

        x = self.norm3(
            x + self.dropout(ff_output)
        )

        return x


# Complete Encoder
class Encoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.layers = nn.ModuleList([
            EncoderLayer()
            for _ in range(NUM_ENCODER_LAYERS)
        ])

    def forward(self, x, source_padding_mask):

        for layer in self.layers:
            x = layer(x, source_padding_mask)

        return x


# Complete Decoder
class Decoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.layers = nn.ModuleList([
            DecoderLayer()
            for _ in range(NUM_DECODER_LAYERS)
        ])

    def forward(
        self,
        x,
        encoder_output,
        target_padding_mask,
        source_padding_mask,
        causal_mask
    ):

        for layer in self.layers:

            x = layer(
                x,
                encoder_output,
                target_padding_mask,
                source_padding_mask,
                causal_mask
            )

        return x


# Complete Transformer
class BasicTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL,
            padding_idx=PAD_IDX
        )

        self.position = PositionalEmbedding(
            D_MODEL,
            MAX_LENGTH
        )

        self.encoder = Encoder()
        self.decoder = Decoder()

        self.dropout = nn.Dropout(DROPOUT)

        self.output_layer = nn.Linear(
            D_MODEL,
            VOCAB_SIZE
        )

    def encode(self, source):

        source_padding_mask = source.eq(PAD_IDX)

        x = self.embedding(source) * math.sqrt(D_MODEL)
        x = x + self.position(source)
        x = self.dropout(x)

        encoder_output = self.encoder(
            x,
            source_padding_mask
        )

        return encoder_output, source_padding_mask

    def decode(
        self,
        target,
        encoder_output,
        source_padding_mask
    ):

        target_padding_mask = target.eq(PAD_IDX)

        x = self.embedding(target) * math.sqrt(D_MODEL)
        x = x + self.position(target)
        x = self.dropout(x)

        target_length = target.size(1)

        causal_mask = torch.triu(
            torch.ones(
                target_length,
                target_length,
                dtype=torch.bool,
                device=target.device
            ),
            diagonal=1
        )

        decoder_output = self.decoder(
            x,
            encoder_output,
            target_padding_mask,
            source_padding_mask,
            causal_mask
        )

        return self.output_layer(decoder_output)

    def forward(self, source, target):

        encoder_output, source_padding_mask = self.encode(
            source
        )

        return self.decode(
            target,
            encoder_output,
            source_padding_mask
        )


# Create model
basic_model = BasicTransformer().to(device)

total_parameters = sum(
    parameter.numel()
    for parameter in basic_model.parameters()
    if parameter.requires_grad
)

print("Basic Transformer created successfully!")
print("Architecture: Encoder-Decoder")
print("Encoder layers:", NUM_ENCODER_LAYERS)
print("Decoder layers:", NUM_DECODER_LAYERS)
print("Attention heads:", NUM_HEADS)
print("Embedding dimension:", D_MODEL)
print("Trainable parameters:", f"{total_parameters:,}")
print("Device:", device)

Basic Transformer created successfully!
Architecture: Encoder-Decoder
Encoder layers: 3
Decoder layers: 3
Attention heads: 4
Embedding dimension: 256
Trainable parameters: 12,178,048
Device: cuda


In [26]:
# Loss function
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

# Optimizer
optimizer = torch.optim.AdamW(
    basic_model.parameters(),
    lr=0.0003,
    weight_decay=0.0001
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)


def train_one_epoch():

    basic_model.train()

    total_loss = 0

    for source, target in train_loader:

        source = source.to(device)
        target = target.to(device)

        # Teacher forcing:
        # Input: BOS, word1, word2...
        # Target: word1, word2, EOS...

        target_input = target[:, :-1]
        target_output = target[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        predictions = basic_model(
            source,
            target_input
        )

        loss = criterion(
            predictions.reshape(-1, VOCAB_SIZE),
            target_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            basic_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(1, len(train_loader))


def validate_model():

    basic_model.eval()

    total_loss = 0

    with torch.no_grad():

        for source, target in validation_loader:

            source = source.to(device)
            target = target.to(device)

            target_input = target[:, :-1]
            target_output = target[:, 1:]

            predictions = basic_model(
                source,
                target_input
            )

            loss = criterion(
                predictions.reshape(-1, VOCAB_SIZE),
                target_output.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / max(1, len(validation_loader))


print("Training configuration ready!")
print("Optimizer: AdamW")
print("Learning rate: 0.0003")
print("Loss: Cross Entropy")
print("Teacher forcing: Enabled")
print("Gradient clipping: 1.0")

Training configuration ready!
Optimizer: AdamW
Learning rate: 0.0003
Loss: Cross Entropy
Teacher forcing: Enabled
Gradient clipping: 1.0


In [33]:
EPOCHS = 8

best_validation_loss = float("inf")

training_history = []

training_start_time = time.time()

print("Starting Basic Transformer training...\n")

for epoch in range(EPOCHS):

    epoch_start = time.time()

    train_loss = train_one_epoch()

    validation_loss = validate_model()

    scheduler.step(validation_loss)

    epoch_time = time.time() - epoch_start

    training_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "time_seconds": epoch_time
    })

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Training Loss: {train_loss:.4f}")
    print(f"Validation Loss: {validation_loss:.4f}")
    print(f"Epoch Time: {epoch_time:.2f} seconds")

    if validation_loss < best_validation_loss:

        best_validation_loss = validation_loss

        torch.save(
            basic_model.state_dict(),
            "basic_transformer_best.pth"
        )

        print("Best model checkpoint saved!")

total_training_time = time.time() - training_start_time

# Load best checkpoint
basic_model.load_state_dict(
    torch.load(
        "basic_transformer_best.pth",
        map_location=device,
        weights_only=True
    )
)

print("\nTraining completed!")

print(
    "Total training time:",
    round(total_training_time / 60, 2),
    "minutes"
)

print(
    "Best validation loss:",
    round(best_validation_loss, 4)
)

print("\nTraining history:")

for record in training_history:
    print(record)

Starting Basic Transformer training...


Epoch 1/8
Training Loss: 1.3029
Validation Loss: 1.1911
Epoch Time: 86.78 seconds
Best model checkpoint saved!

Epoch 2/8
Training Loss: 1.1428
Validation Loss: 1.0865
Epoch Time: 87.74 seconds
Best model checkpoint saved!

Epoch 3/8
Training Loss: 1.0202
Validation Loss: 0.9972
Epoch Time: 78.47 seconds
Best model checkpoint saved!

Epoch 4/8
Training Loss: 0.9189
Validation Loss: 0.9320
Epoch Time: 77.71 seconds
Best model checkpoint saved!

Epoch 5/8
Training Loss: 0.8340
Validation Loss: 0.8893
Epoch Time: 77.66 seconds
Best model checkpoint saved!

Epoch 6/8
Training Loss: 0.7661
Validation Loss: 0.8404
Epoch Time: 77.89 seconds
Best model checkpoint saved!

Epoch 7/8
Training Loss: 0.7050
Validation Loss: 0.8135
Epoch Time: 77.62 seconds
Best model checkpoint saved!

Epoch 8/8
Training Loss: 0.6542
Validation Loss: 0.7812
Epoch Time: 75.04 seconds
Best model checkpoint saved!

Training completed!
Total training time: 10.7 minutes
Best vali

In [2]:
# Load best model
basic_model.eval()


# Translation function
def basic_translate(text):

    basic_model.eval()

    source_ids = encode_sentence(text)

    source = torch.tensor(
        [source_ids],
        dtype=torch.long
    ).to(device)

    start_time = time.time()

    with torch.no_grad():

        encoder_output, source_padding_mask = (
            basic_model.encode(source)
        )

        generated = torch.tensor(
            [[BOS_IDX]],
            dtype=torch.long,
            device=device
        )

        for _ in range(MAX_LENGTH - 1):

            logits = basic_model.decode(
                generated,
                encoder_output,
                source_padding_mask
            )

            next_token = logits[:, -1, :].argmax(
                dim=-1
            )

            generated = torch.cat(
                [generated, next_token.unsqueeze(1)],
                dim=1
            )

            if next_token.item() == EOS_IDX:
                break

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time = time.time() - start_time

    result = decode_sentence(
        generated[0].tolist()
    )

    return result, inference_time


# Test examples
test_sentences = [
    "Hello how are you",
    "India is a beautiful country",
    "Education is important",
    "Artificial intelligence is changing the world",
    "Machine learning is a part of artificial intelligence"
]

print("BASIC TRANSFORMER TRANSLATIONS\n")

for sentence in test_sentences:

    translated, elapsed = basic_translate(sentence)

    print("English:", sentence)
    print("Hindi:", translated)
    print("Inference time:", round(elapsed, 4), "seconds")
    print("-" * 60)


# BLEU and chrF evaluation
!pip install -q sacrebleu

import sacrebleu

predictions = []
references = []

inference_times = []

print("\nEvaluating test dataset...")

for i in range(len(basic_test)):

    item = basic_test[i]

    english = item["en"]
    hindi_reference = item["hi"]

    translated, elapsed = basic_translate(english)

    predictions.append(translated)
    references.append(hindi_reference)

    inference_times.append(elapsed)

    if (i + 1) % 100 == 0:
        print("Evaluated:", i + 1)


bleu_score = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

chrf_score = sacrebleu.corpus_chrf(
    predictions,
    [references]
)

average_inference_time = np.mean(inference_times)

print("\n========== BASIC TRANSFORMER RESULTS ==========")

print("BLEU Score:", round(bleu_score.score, 4))
print("chrF Score:", round(chrf_score.score, 4))
print(
    "Average inference time:",
    round(average_inference_time, 5),
    "seconds"
)

print("Training time:", round(total_training_time, 2), "seconds")

print("Trainable parameters:", f"{total_parameters:,}")


# Save model and vocabulary
SAVE_DIR = "/content/drive/MyDrive/Basic_Transformer_Results"

os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(
    basic_model.state_dict(),
    os.path.join(SAVE_DIR, "basic_transformer.pth")
)

with open(
    os.path.join(SAVE_DIR, "vocabulary.json"),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        vocab,
        file,
        ensure_ascii=False
    )


# Save metrics
basic_results = {
    "model": "Basic Transformer",
    "architecture": "Encoder-Decoder",
    "trainable_parameters": total_parameters,
    "training_time_seconds": total_training_time,
    "average_inference_time_seconds": float(
        average_inference_time
    ),
    "BLEU": bleu_score.score,
    "chrF": chrf_score.score,
    "best_validation_loss": best_validation_loss,
    "epochs": EPOCHS,
    "training_samples": len(basic_train),
    "validation_samples": len(basic_validation),
    "test_samples": len(basic_test)
}

with open(
    os.path.join(SAVE_DIR, "basic_transformer_metrics.json"),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        basic_results,
        file,
        indent=4,
        ensure_ascii=False
    )


# Save translation examples
with open(
    os.path.join(SAVE_DIR, "translation_examples.txt"),
    "w",
    encoding="utf-8"
) as file:

    for i in range(min(100, len(predictions))):

        file.write("English: " + basic_test[i]["en"] + "\n")
        file.write("Reference: " + references[i] + "\n")
        file.write("Prediction: " + predictions[i] + "\n")
        file.write("-" * 60 + "\n")


print("\nAll Basic Transformer results saved!")
print("Folder:", SAVE_DIR)

print("\nSaved files:")

for filename in os.listdir(SAVE_DIR):
    print(filename)

NameError: name 'basic_model' is not defined

In [3]:
import torch
import os

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Check whether architecture exists
if "BasicTransformer" not in globals():
    print("BasicTransformer class is missing.")
    print("Rerun Cell 2 (architecture).")

else:
    # Recreate model
    basic_model = BasicTransformer().to(device)

    # Restore trained checkpoint if available
    if os.path.exists("basic_transformer_best.pth"):

        basic_model.load_state_dict(
            torch.load(
                "basic_transformer_best.pth",
                map_location=device,
                weights_only=True
            )
        )

        print("Trained checkpoint restored!")

    else:
        print("Checkpoint not found.")
        print("Training must be repeated.")

    basic_model.eval()

Device: cpu
BasicTransformer class is missing.
Rerun Cell 2 (architecture).


In [29]:
from transformers import MarianTokenizer
import torch
import torch.nn as nn

# Load tokenizer only, NOT the pretrained translation model
basic_sp_tokenizer = MarianTokenizer.from_pretrained(
    "Helsinki-NLP/opus-mt-en-hi"
)

# Special token IDs
PAD_IDX = basic_sp_tokenizer.pad_token_id
EOS_IDX = basic_sp_tokenizer.eos_token_id
UNK_IDX = basic_sp_tokenizer.unk_token_id

# Create a separate learnable BOS token
BOS_IDX = len(basic_sp_tokenizer)

# Updated vocabulary size
VOCAB_SIZE = len(basic_sp_tokenizer) + 1

print("SentencePiece tokenizer loaded!")
print("Vocabulary size:", VOCAB_SIZE)
print("PAD:", PAD_IDX)
print("EOS:", EOS_IDX)
print("BOS:", BOS_IDX)
print("UNK:", UNK_IDX)


# Replace encoding function
def encode_sentence(text):

    token_ids = basic_sp_tokenizer(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH - 2
    )["input_ids"]

    return [BOS_IDX] + token_ids + [EOS_IDX]


# Replace decoding function
def decode_sentence(ids):

    cleaned_ids = []

    for idx in ids:

        idx = int(idx)

        if idx == EOS_IDX:
            break

        if idx in (PAD_IDX, BOS_IDX):
            continue

        if idx < len(basic_sp_tokenizer):
            cleaned_ids.append(idx)

    return basic_sp_tokenizer.decode(
        cleaned_ids,
        skip_special_tokens=True
    )


# Verify tokenization
example = "India is a beautiful country"

encoded = encode_sentence(example)

print("\nOriginal:", example)
print("Encoded:", encoded)
print("Decoded:", decode_sentence(encoded))

SentencePiece tokenizer loaded!
Vocabulary size: 61951
PAD: 61949
EOS: 0
BOS: 61950
UNK: 1

Original: India is a beautiful country
Encoded: [61950, 4535, 23, 19, 3511, 2126, 0]
Decoded: India is a beautiful country


/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [30]:
# Reinitialize model with the new vocabulary
basic_model = BasicTransformer().to(device)

# Reinitialize optimizer
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

optimizer = torch.optim.AdamW(
    basic_model.parameters(),
    lr=0.0003,
    weight_decay=0.0001
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

total_parameters = sum(
    p.numel()
    for p in basic_model.parameters()
    if p.requires_grad
)

print("Transformer reinitialized!")
print("New vocabulary:", VOCAB_SIZE)
print("New parameters:", f"{total_parameters:,}")

Transformer reinitialized!
New vocabulary: 61951
New parameters: 35,750,911


In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')

for folder in [
    "/content/drive/MyDrive/Basic_Transformer_Results",
    "/content/drive/MyDrive/English_Hindi_Transformer"
]:
    print("\nChecking:", folder)

    if os.path.exists(folder):
        for filename in os.listdir(folder):
            print(" -", filename)
    else:
        print("Folder not found")

Mounted at /content/drive

Checking: /content/drive/MyDrive/Basic_Transformer_Results
 - basic_transformer.pth
 - vocabulary.json
 - basic_transformer_metrics.json
 - translation_examples.txt

Checking: /content/drive/MyDrive/English_Hindi_Transformer
 - config.json
 - generation_config.json
 - model.safetensors
 - tokenizer_config.json
 - vocab.json
 - source.spm
 - target.spm
 - training_args.bin
 - training_results.txt


In [5]:
!pip install -q transformers datasets sentencepiece sacrebleu tqdm

In [6]:
import os
import gc
import json
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from transformers import AutoTokenizer
from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm
from google.colab import drive

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Training will be slower.")

# Mount Google Drive
drive.mount("/content/drive")

# Separate folder so old results are not overwritten
SAVE_DIR = "/content/drive/MyDrive/Basic_Transformer_Scratch"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Save directory:", SAVE_DIR)

Device: cpu
Running on CPU. Training will be slower.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save directory: /content/drive/MyDrive/Basic_Transformer_Scratch


In [7]:
print("Loading IIT Bombay English-Hindi dataset...")

raw_dataset = load_dataset(
    "cfilt/iitb-english-hindi",
    split="train"
)

print("Total available rows:", len(raw_dataset))

# Shuffle and select a manageable subset
N_SAMPLES = min(6000, len(raw_dataset))

subset = raw_dataset.shuffle(
    seed=SEED
).select(range(N_SAMPLES))

pairs = []

for row in tqdm(subset, desc="Cleaning dataset"):

    translation = row["translation"]

    english = str(translation.get("en", "")).strip()
    hindi = str(translation.get("hi", "")).strip()

    if not english or not hindi:
        continue

    # Remove excessively long sentences
    if len(english) > 400 or len(hindi) > 400:
        continue

    pairs.append({
        "en": english,
        "hi": hindi
    })

random.shuffle(pairs)

# 80% train, 10% validation, 10% test
n = len(pairs)

train_end = int(n * 0.80)
val_end = int(n * 0.90)

train_pairs = pairs[:train_end]
val_pairs = pairs[train_end:val_end]
test_pairs = pairs[val_end:]

print("\nDATASET SUMMARY")
print("Total usable pairs:", n)
print("Training:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Testing:", len(test_pairs))

print("\nExample:")
print("English:", train_pairs[0]["en"])
print("Hindi:", train_pairs[0]["hi"])

# Save dataset statistics
dataset_info = {
    "dataset": "IIT Bombay English-Hindi",
    "total_pairs": n,
    "training_pairs": len(train_pairs),
    "validation_pairs": len(val_pairs),
    "testing_pairs": len(test_pairs)
}

with open(
    os.path.join(SAVE_DIR, "dataset_info.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=4)

Loading IIT Bombay English-Hindi dataset...


README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.7kB            

... (5 outputs left out)

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

Total available rows: 1659083


Cleaning dataset:   0%|          | 0/6000 [00:00<?, ?it/s]


DATASET SUMMARY
Total usable pairs: 5915
Training: 4732
Validation: 591
Testing: 592

Example:
English: Many years after Chatrik 's essay on Queen Victoria, Lytton Strachey wrote a new kind of biography of the queen.
Hindi: महारानी विक्टोरिया की एक इसी प्रकार की जीवनी लिखी।


In [8]:
print("Loading SentencePiece tokenizer...")

TOKENIZER_NAME = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

PAD_IDX = tokenizer.pad_token_id
EOS_IDX = tokenizer.eos_token_id
UNK_IDX = tokenizer.unk_token_id

# Additional BOS token, outside the tokenizer's vocabulary
BOS_IDX = len(tokenizer)

VOCAB_SIZE = len(tokenizer) + 1

MAX_LENGTH = 64
BATCH_SIZE = 32

print("Vocabulary size:", VOCAB_SIZE)
print("PAD:", PAD_IDX)
print("EOS:", EOS_IDX)
print("BOS:", BOS_IDX)


def encode_sentence(text):

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH - 2
    )["input_ids"]

    return [BOS_IDX] + token_ids + [EOS_IDX]


class TranslationDataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        row = self.data[index]

        source_ids = encode_sentence(row["en"])
        target_ids = encode_sentence(row["hi"])

        return (
            torch.tensor(source_ids, dtype=torch.long),
            torch.tensor(target_ids, dtype=torch.long)
        )


def collate_batch(batch):

    sources = [item[0] for item in batch]
    targets = [item[1] for item in batch]

    sources = pad_sequence(
        sources,
        batch_first=True,
        padding_value=PAD_IDX
    )

    targets = pad_sequence(
        targets,
        batch_first=True,
        padding_value=PAD_IDX
    )

    return sources, targets


train_dataset = TranslationDataset(train_pairs)
val_dataset = TranslationDataset(val_pairs)
test_dataset = TranslationDataset(test_pairs)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

print("\nDataLoaders created successfully!")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Loading SentencePiece tokenizer...


config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Vocabulary size: 61951
PAD: 61949
EOS: 0
BOS: 61950

DataLoaders created successfully!
Training batches: 148
Validation batches: 19
Testing batches: 19


/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [9]:
class BasicTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=128,
        nhead=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        dim_feedforward=512,
        dropout=0.1,
        max_length=MAX_LENGTH
    ):

        super().__init__()

        self.d_model = d_model

        # Shared token embedding
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_IDX
        )

        # Learnable positional embeddings
        self.position_embedding = nn.Embedding(
            max_length,
            d_model
        )

        # Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )

        # Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_decoder_layers
        )

        self.dropout = nn.Dropout(dropout)

        # Output projection tied to shared token embeddings
        self.output_layer = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        self.output_layer.weight = self.token_embedding.weight

    def create_embeddings(self, token_ids):

        batch_size, seq_length = token_ids.shape

        positions = torch.arange(
            seq_length,
            device=token_ids.device
        ).unsqueeze(0).expand(batch_size, -1)

        embeddings = (
            self.token_embedding(token_ids)
            * (self.d_model ** 0.5)
        )

        embeddings = embeddings + self.position_embedding(positions)

        return self.dropout(embeddings)

    def forward(self, source, target):

        # Padding masks
        source_padding_mask = source.eq(PAD_IDX)
        target_padding_mask = target.eq(PAD_IDX)

        # Causal mask prevents decoder from seeing future tokens
        target_length = target.size(1)

        causal_mask = torch.triu(
            torch.ones(
                target_length,
                target_length,
                device=target.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # Encode English input
        source_embeddings = self.create_embeddings(source)

        encoder_output = self.encoder(
            source_embeddings,
            src_key_padding_mask=source_padding_mask
        )

        # Decode Hindi output
        target_embeddings = self.create_embeddings(target)

        decoder_output = self.decoder(
            tgt=target_embeddings,
            memory=encoder_output,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask
        )

        logits = self.output_layer(decoder_output)

        return logits


# Create model from random initialization
basic_model = BasicTransformer(
    vocab_size=VOCAB_SIZE
).to(device)

total_parameters = sum(
    parameter.numel()
    for parameter in basic_model.parameters()
    if parameter.requires_grad
)

print("Basic Transformer initialized successfully!")
print("Architecture: Encoder-Decoder")
print("Trainable parameters:", f"{total_parameters:,}")

# Loss and optimizer
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

optimizer = torch.optim.AdamW(
    basic_model.parameters(),
    lr=0.0003,
    weight_decay=0.0001
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

Basic Transformer initialized successfully!
Architecture: Encoder-Decoder
Trainable parameters: 8,863,616


In [10]:
EPOCHS = 3

best_val_loss = float("inf")
history = []

training_start = time.time()


def run_validation():

    basic_model.eval()

    total_loss = 0
    total_batches = 0

    with torch.no_grad():

        for source, target in val_loader:

            source = source.to(device)
            target = target.to(device)

            decoder_input = target[:, :-1]
            expected = target[:, 1:]

            logits = basic_model(
                source,
                decoder_input
            )

            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                expected.reshape(-1)
            )

            total_loss += loss.item()
            total_batches += 1

    return total_loss / max(total_batches, 1)


for epoch in range(EPOCHS):

    basic_model.train()

    total_train_loss = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for source, target in progress:

        source = source.to(device)
        target = target.to(device)

        decoder_input = target[:, :-1]
        expected = target[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        logits = basic_model(
            source,
            decoder_input
        )

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            expected.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            basic_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_train_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = total_train_loss / len(train_loader)

    val_loss = run_validation()

    scheduler.step(val_loss)

    epoch_info = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": val_loss
    }

    history.append(epoch_info)

    print(f"\nEpoch {epoch + 1}")
    print("Training loss:", round(train_loss, 4))
    print("Validation loss:", round(val_loss, 4))

    # Save best model after each epoch
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            basic_model.state_dict(),
            os.path.join(SAVE_DIR, "basic_transformer_best.pth")
        )

        print("Best checkpoint saved!")

    # Save training history after every epoch
    with open(
        os.path.join(SAVE_DIR, "training_history.json"),
        "w"
    ) as f:
        json.dump(history, f, indent=4)


training_time = time.time() - training_start

print("\nTraining completed!")
print("Total training time:", round(training_time, 2), "seconds")
print("Best validation loss:", round(best_val_loss, 4))

Epoch 1/3:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(



Epoch 1
Training loss: 16.5661
Validation loss: 8.5681
Best checkpoint saved!


Epoch 2/3:   0%|          | 0/148 [00:00<?, ?it/s]


Epoch 2
Training loss: 7.3102
Validation loss: 5.101
Best checkpoint saved!


Epoch 3/3:   0%|          | 0/148 [00:00<?, ?it/s]


Epoch 3
Training loss: 4.9299
Validation loss: 3.9829
Best checkpoint saved!

Training completed!
Total training time: 2496.45 seconds
Best validation loss: 3.9829


In [11]:
# Load best checkpoint
checkpoint_path = os.path.join(
    SAVE_DIR,
    "basic_transformer_best.pth"
)

basic_model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

basic_model.eval()

print("Best model loaded!")


def basic_translate(text):

    basic_model.eval()

    source_ids = encode_sentence(text)

    source = torch.tensor(
        [source_ids],
        dtype=torch.long,
        device=device
    )

    generated = [BOS_IDX]

    with torch.no_grad():

        for _ in range(MAX_LENGTH - 1):

            target = torch.tensor(
                [generated],
                dtype=torch.long,
                device=device
            )

            logits = basic_model(
                source,
                target
            )

            next_token = logits[0, -1].argmax().item()

            if next_token == EOS_IDX:
                break

            if next_token == PAD_IDX:
                break

            generated.append(next_token)

    return tokenizer.decode(
        [
            token_id
            for token_id in generated[1:]
            if token_id < len(tokenizer)
        ],
        skip_special_tokens=True
    ).strip()


# Evaluate a manageable test subset
EVAL_SIZE = min(200, len(test_pairs))

evaluation_pairs = test_pairs[:EVAL_SIZE]

predictions = []
references = []
translation_records = []

total_inference_time = 0

print("\nGenerating translations...")

for row in tqdm(evaluation_pairs):

    english = row["en"]
    hindi_reference = row["hi"]

    start = time.time()

    hindi_prediction = basic_translate(english)

    elapsed = time.time() - start

    total_inference_time += elapsed

    predictions.append(hindi_prediction)
    references.append(hindi_reference)

    translation_records.append({
        "english": english,
        "reference": hindi_reference,
        "prediction": hindi_prediction,
        "inference_time": elapsed
    })


# BLEU and chrF
bleu_metric = BLEU()
chrf_metric = CHRF()

bleu_score = bleu_metric.corpus_score(
    predictions,
    [references]
).score

chrf_score = chrf_metric.corpus_score(
    predictions,
    [references]
).score

average_inference_time = (
    total_inference_time / max(EVAL_SIZE, 1)
)

# Final metrics
metrics = {
    "model": "Basic Transformer",
    "architecture": "Encoder-Decoder",
    "training_samples": len(train_pairs),
    "validation_samples": len(val_pairs),
    "test_samples": len(test_pairs),
    "evaluated_samples": EVAL_SIZE,
    "epochs": EPOCHS,
    "training_time_seconds": training_time,
    "best_validation_loss": best_val_loss,
    "bleu_score": bleu_score,
    "chrf_score": chrf_score,
    "average_inference_time_seconds": average_inference_time,
    "trainable_parameters": total_parameters,
    "device": str(device)
}

print("\n========== BASIC TRANSFORMER RESULTS ==========")

for key, value in metrics.items():
    print(f"{key}: {value}")


# Save metrics
with open(
    os.path.join(SAVE_DIR, "basic_transformer_metrics.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, ensure_ascii=False, indent=4)


# Save translations
with open(
    os.path.join(SAVE_DIR, "translation_examples.txt"),
    "w",
    encoding="utf-8"
) as f:

    for i, record in enumerate(translation_records):

        f.write(f"Example {i + 1}\n")
        f.write(f"English: {record['english']}\n")
        f.write(f"Reference Hindi: {record['reference']}\n")
        f.write(f"Predicted Hindi: {record['prediction']}\n")
        f.write(
            f"Inference time: {record['inference_time']:.4f} seconds\n"
        )
        f.write("-" * 70 + "\n")


# Save tokenizer files for reproducibility
tokenizer.save_pretrained(
    os.path.join(SAVE_DIR, "tokenizer")
)

# Save architecture configuration
config = {
    "vocab_size": VOCAB_SIZE,
    "d_model": 128,
    "nhead": 4,
    "num_encoder_layers": 2,
    "num_decoder_layers": 2,
    "dim_feedforward": 512,
    "max_length": MAX_LENGTH
}

with open(
    os.path.join(SAVE_DIR, "model_config.json"),
    "w"
) as f:
    json.dump(config, f, indent=4)


print("\n========== SAMPLE TRANSLATIONS ==========")

for record in translation_records[:10]:

    print("\nEnglish:", record["english"])
    print("Reference:", record["reference"])
    print("Prediction:", record["prediction"])


print("\nAll results saved successfully!")
print("Folder:", SAVE_DIR)

print("\nSaved files:")

for filename in os.listdir(SAVE_DIR):
    print("-", filename)

Best model loaded!

Generating translations...


  0%|          | 0/200 [00:00<?, ?it/s]


========== BASIC TRANSFORMER RESULTS ==========
model: Basic Transformer
architecture: Encoder-Decoder
training_samples: 4732
validation_samples: 591
test_samples: 592
evaluated_samples: 200
epochs: 3
training_time_seconds: 2496.4512355327606
best_validation_loss: 3.9828918105677555
bleu_score: 0.012908836382327982
chrf_score: 1.9285325365980255
average_inference_time_seconds: 1.1488820648193359
trainable_parameters: 8863616
device: cpu

========== SAMPLE TRANSLATIONS ==========

English: In the fast enveloping darkness of the night, no one member of the pilgrim party knew where others were or what had happened to them.
Reference: रात्रि के बढ़ते हुए अंधकार में यात्रियों के दल के किसी भी सदस्य को एक-दूसरे की स्थिति के बारे में कुछ पता नहीं चलता।
Prediction: ा ा ा ा ा ा ा ा ा ा ा ा ा ा ा ा ा ा ा ा ा

English: Australian dollar
Reference: आस्ट्रेलियाई डॉलर
Prediction: ा ारा का का का का कारा

English: Tender and un - aging.
Reference: प्रेम दर्शानेवाली और समायु;
Prediction: ा ा ा ा ा ा ा

In [12]:
!pip install -q transformers datasets sentencepiece accelerate sacrebleu tqdm

In [13]:
import os
import gc
import json
import time
import random
import torch
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq
)

from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm
from google.colab import drive

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU mode. mT5 training will be slow.")

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/MT5_Results"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Save directory:", SAVE_DIR)

Device: cpu
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save directory: /content/drive/MyDrive/MT5_Results


In [14]:
from datasets import load_dataset

if all(
    name in globals()
    for name in ["train_pairs", "val_pairs", "test_pairs"]
):
    print("Reusing existing Basic Transformer dataset splits.")

else:
    print("Rebuilding deterministic dataset splits...")

    random.seed(SEED)

    raw_dataset = load_dataset(
        "cfilt/iitb-english-hindi",
        split="train"
    )

    subset = raw_dataset.shuffle(
        seed=SEED
    ).select(range(min(6000, len(raw_dataset))))

    pairs = []

    for row in tqdm(subset):

        translation = row["translation"]

        en = str(translation.get("en", "")).strip()
        hi = str(translation.get("hi", "")).strip()

        if en and hi and len(en) <= 400 and len(hi) <= 400:
            pairs.append({"en": en, "hi": hi})

    random.shuffle(pairs)

    n = len(pairs)

    train_end = int(n * 0.80)
    val_end = int(n * 0.90)

    train_pairs = pairs[:train_end]
    val_pairs = pairs[train_end:val_end]
    test_pairs = pairs[val_end:]

print("Training:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Testing:", len(test_pairs))

print("\nExample:")
print("English:", train_pairs[0]["en"])
print("Hindi:", train_pairs[0]["hi"])

Reusing existing Basic Transformer dataset splits.
Training: 4732
Validation: 591
Testing: 592

Example:
English: Many years after Chatrik 's essay on Queen Victoria, Lytton Strachey wrote a new kind of biography of the queen.
Hindi: महारानी विक्टोरिया की एक इसी प्रकार की जीवनी लिखी।


In [15]:
MODEL_NAME = "google/mt5-small"

mt5_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

class MT5Dataset(Dataset):

    def __init__(self, data):

        self.data = data

    def __len__(self):

        return len(self.data)

    def __getitem__(self, index):

        row = self.data[index]

        source_text = (
            "translate English to Hindi: " + row["en"]
        )

        target_text = row["hi"]

        source = mt5_tokenizer(
            source_text,
            max_length=MAX_SOURCE_LENGTH,
            truncation=True
        )

        target = mt5_tokenizer(
            text_target=target_text,
            max_length=MAX_TARGET_LENGTH,
            truncation=True
        )

        return {
            "input_ids": source["input_ids"],
            "attention_mask": source["attention_mask"],
            "labels": target["input_ids"]
        }


train_mt5 = MT5Dataset(train_pairs)
val_mt5 = MT5Dataset(val_pairs)
test_mt5 = MT5Dataset(test_pairs)

# Dynamic padding and ignored label padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=mt5_tokenizer,
    model=None,
    label_pad_token_id=-100,
    return_tensors="pt"
)

BATCH_SIZE = 2 if device.type == "cpu" else 4

train_loader_mt5 = DataLoader(
    train_mt5,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_loader_mt5 = DataLoader(
    val_mt5,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

print("mT5 datasets prepared!")
print("Training batches:", len(train_loader_mt5))
print("Validation batches:", len(val_loader_mt5))

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

mT5 datasets prepared!
Training batches: 2366
Validation batches: 296


In [16]:
print("Loading pretrained mT5-small...")

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

mt5_model = mt5_model.to(device)

total_parameters_mt5 = sum(
    p.numel()
    for p in mt5_model.parameters()
)

trainable_parameters_mt5 = sum(
    p.numel()
    for p in mt5_model.parameters()
    if p.requires_grad
)

print("\nmT5 loaded successfully!")

print(
    "Total parameters:",
    f"{total_parameters_mt5:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters_mt5:,}"
)

print("Architecture: Encoder-Decoder")
print("Pretrained weights: Yes")

Loading pretrained mT5-small...


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


mT5 loaded successfully!
Total parameters: 300,176,768
Trainable parameters: 300,176,768
Architecture: Encoder-Decoder
Pretrained weights: Yes


In [17]:
EPOCHS_MT5 = 1

if device.type == "cpu":

    print("CPU mode: limiting training to 300 examples.")

    cpu_train_pairs = train_pairs[:300]

    train_mt5 = MT5Dataset(cpu_train_pairs)

    train_loader_mt5 = DataLoader(
        train_mt5,
        batch_size=2,
        shuffle=True,
        collate_fn=data_collator
    )

    EPOCHS_MT5 = 1

optimizer_mt5 = torch.optim.AdamW(
    mt5_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

best_val_loss_mt5 = float("inf")
mt5_history = []

training_start_mt5 = time.time()

for epoch in range(EPOCHS_MT5):

    mt5_model.train()

    total_train_loss = 0

    progress = tqdm(
        train_loader_mt5,
        desc=f"mT5 Epoch {epoch + 1}"
    )

    for batch in progress:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer_mt5.zero_grad(set_to_none=True)

        outputs = mt5_model(**batch)

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            mt5_model.parameters(),
            1.0
        )

        optimizer_mt5.step()

        total_train_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = (
        total_train_loss / max(len(train_loader_mt5), 1)
    )

    # Validation
    mt5_model.eval()

    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader_mt5:

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = mt5_model(**batch)

            total_val_loss += outputs.loss.item()

    val_loss = (
        total_val_loss / max(len(val_loader_mt5), 1)
    )

    print(f"\nEpoch {epoch + 1}")
    print("Training loss:", round(train_loss, 4))
    print("Validation loss:", round(val_loss, 4))

    mt5_history.append({
        "epoch": epoch + 1,
        "training_loss": train_loss,
        "validation_loss": val_loss
    })

    if val_loss < best_val_loss_mt5:

        best_val_loss_mt5 = val_loss

        mt5_model.save_pretrained(SAVE_DIR)
        mt5_tokenizer.save_pretrained(SAVE_DIR)

        print("Best mT5 checkpoint saved!")

training_time_mt5 = time.time() - training_start_mt5

with open(
    os.path.join(SAVE_DIR, "training_history.json"),
    "w"
) as f:

    json.dump(mt5_history, f, indent=4)

print("\nFine-tuning completed!")
print("Training time:", round(training_time_mt5, 2), "seconds")
print("Best validation loss:", best_val_loss_mt5)

CPU mode: limiting training to 300 examples.


mT5 Epoch 1:   0%|          | 0/150 [00:00<?, ?it/s]


Epoch 1
Training loss: 24.3577
Validation loss: 15.1345


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best mT5 checkpoint saved!

Fine-tuning completed!
Training time: 1886.3 seconds
Best validation loss: 15.134494203167993


In [18]:
import os

RESULTS_DIR = "/content/drive/MyDrive/English_Hindi_Final/results"

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Results folder created:", RESULTS_DIR)

Results folder created: /content/drive/MyDrive/English_Hindi_Final/results


In [20]:
import os

from google.colab import drive
drive.mount('/content/drive')

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith((".json", ".csv", ".txt")):
            if any(word in file.lower() for word in [
                "basic", "transformer", "mt5", "metric", "result"
            ]):
                print(os.path.join(root, file))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ATLAB/230911294/LAB5EX1/build (1)/3f8049328ae203f0ddf748ac7582a8c6/dart_build_result.json
/content/drive/MyDrive/ATLAB/230911294/LAB5EX1/build/3f8049328ae203f0ddf748ac7582a8c6/dart_build_result.json
/content/drive/MyDrive/ATLAB/230911294/LAB5EX3/build (1)/fcc66af2d14464bac6c0a01cce2a9e31/dart_build_result.json
/content/drive/MyDrive/ATLAB/230911294/LAB5EX3/build/fcc66af2d14464bac6c0a01cce2a9e31/dart_build_result.json
/content/drive/MyDrive/ATLAB/230911294/LAB4EX9/build/ce2293a1eff65caa313d3e621950da3c/dart_build_result.json
/content/drive/MyDrive/ATLAB/230911294/LAB5EX4/.dart_tool (1)/flutter_build/9fdef4112b6fa5b01e0b2b760cb0fa8d/dart_build_result.json
/content/drive/MyDrive/ATLAB/230911294/LAB5EX4/.dart_tool/flutter_build/9fdef4112b6fa5b01e0b2b760cb0fa8d/dart_build_result.json
/content/drive/MyDrive/ATLAB/230911294/LAB5EX4/build/6dc73

KeyboardInterrupt: 